# Álbum x4 en Google Colab

Trabajo Integrador 2 · IFTS 24 · Subtarea A

Ampliá fotos pequeñas con Stable Diffusion x4 y revisá qué cambió con RT-DETR y un mapa de diferencias.

Activá una **GPU** en **Entorno de ejecución → Cambiar tipo de entorno de ejecución**. Ejecutá las celdas en orden. La interfaz de Gradio aparece debajo de la última celda. Usá las fotos públicas de ejemplo para la demo.

## 1. Cargar el proyecto

Dejá `REPOSITORIO` vacío para subir el ZIP del proyecto desde tu equipo. El ZIP debe incluir `app.py`, `uv.lock`, `src/` y `assets/`. Cuando publiques el proyecto, también podés indicar la URL de tu repositorio público de GitHub.

In [ ]:
import re
import subprocess
import tempfile
from pathlib import Path
from zipfile import ZipFile

from google.colab import files

REPOSITORIO = ""  # Ejemplo: https://github.com/tu-usuario/album-x4

carpeta = Path(tempfile.mkdtemp(prefix="album-x4-", dir="/content"))
if REPOSITORIO.strip():
    url = REPOSITORIO.strip().rstrip("/")
    if not re.fullmatch(r"https://github\.com/[\w.-]+/[\w.-]+", url):
        raise ValueError("Ingresá la URL del repositorio público de GitHub.")
    proyecto = carpeta / "proyecto"
    subprocess.run(["git", "clone", "--depth", "1", url, str(proyecto)], check=True)
else:
    files.upload(target_dir=str(carpeta))
    archivos_zip = [ruta for ruta in carpeta.iterdir() if ruta.suffix.lower() == ".zip"]
    if len(archivos_zip) != 1:
        raise ValueError("Subí un solo archivo ZIP con el proyecto completo.")
    with ZipFile(archivos_zip[0]) as archivo_zip:
        archivo_zip.extractall(carpeta)
    candidatos = [
        ruta.parent for ruta in carpeta.rglob("app.py")
        if (ruta.parent / "pyproject.toml").is_file()
        and (ruta.parent / "uv.lock").is_file()
        and (ruta.parent / "assets" / "ejemplos").is_dir()
    ]
    if len(candidatos) != 1:
        raise ValueError("El ZIP debe contener una carpeta del proyecto con app.py, uv.lock y assets.")
    proyecto = candidatos[0]

print("Proyecto listo en", proyecto)

## 2. Instalar el entorno y comprobar la GPU

`uv` prepara Python 3.12 y usa las versiones fijadas en `uv.lock`, incluido PyTorch para CUDA. La primera instalación tarda unos minutos. Guardá el nombre de la GPU y su VRAM para registrar tu prueba en el README.

In [ ]:
import shutil
import sys

def ejecutar_y_mostrar(comando, **opciones):
    resultado = subprocess.run(
        comando, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", errors="replace", **opciones,
    )
    print(resultado.stdout, end="", flush=True)
    resultado.check_returncode()

print("Comprobando la GPU...", flush=True)
ejecutar_y_mostrar(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"])
ejecutar_y_mostrar([sys.executable, "-m", "pip", "install", "-q", "uv==0.12.10"])
uv = shutil.which("uv")
if uv is None:
    raise RuntimeError("Colab no encontró uv después de instalarlo.")

print("Preparando Python 3.12 y las dependencias. Puede tardar unos minutos.", flush=True)
ejecutar_y_mostrar([uv, "sync", "--locked"], cwd=proyecto)
ejecutar_y_mostrar([uv, "run", "--locked", "app.py", "--entorno"], cwd=proyecto)
print("Entorno listo. Ejecutá la celda 3.")

## 3. Usar Gradio dentro de Colab

Ejecutá la celda y esperá a que aparezca **Álbum x4** debajo del código. Usá los controles en ese panel. La celda sigue activa mientras la app está disponible. Si el arranque falla, el mensaje aparece en la salida de la celda.

Elegí una foto de ejemplo y hacé clic en **Ampliar foto**. En el primer uso se descargan los pesos de Hugging Face. Para probar el retrato `Vintage_female_photograph_2.jpg`, empezá con **Más detalle**, intensidad **20**. Mirá **Lado a lado** y usá **Suave** con intensidad **10** para repetir el ajuste de la versión inicial. El par de Edward Curtis incluye una referencia de alta resolución, descrita en `assets/FUENTES.md`.

In [ ]:
from IPython.display import IFrame, display

print("Iniciando Gradio dentro de Colab...", flush=True)
comando = [uv, "run", "--locked", "python", "-u", "app.py", "--colab"]
with subprocess.Popen(
    comando, cwd=proyecto, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, encoding="utf-8", errors="replace",
) as proceso:
    try:
        panel_mostrado = False
        for linea in proceso.stdout:
            print(linea, end="", flush=True)
            if not panel_mostrado and "public URL" in linea:
                enlace = re.search(r"https://[a-zA-Z0-9-]+[.]gradio[.]live/?", linea)
                if enlace:
                    display(IFrame(enlace.group(0), width="100%", height=900, title="Álbum x4"))
                    panel_mostrado = True
        codigo = proceso.wait()
        if codigo != 0:
            raise RuntimeError(f"Gradio terminó con código {codigo}. Revisá el mensaje de arriba.")
    finally:
        if proceso.poll() is None:
            proceso.terminate()